# LightGBM hyper-parameter tuning 

Starting point: the reference model `lgbm_otherdir` with the current `PARAMS` of `src/models/lgbm.py`.

Each configuration is a full walk-forward.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")

import numpy as np
import pandas as pd

from src.data.data import build_auctions
from src.evaluation import Evaluator
from src.features.build_features import (
    LAG_FEATURES, OTHER_DIRECTION_FEATURES,
    add_lag_features, add_model_features, add_other_direction_features, build_features,
)
from src.models import ridge
from src.models.baseline import PreviousKnownAuctionBaseline
from src.models.lgbm import PARAMS
from src.models.tuning import CONFIGS, bidding_summary, run_configs, summarise, WEIGHTING, WINDOWS, per_year_report, run_variants

BEST = "background-color: #c6efce; color: #006100; font-weight: bold"

def mean_gap_to_best(table, models_in="rows", how="min"):
    t = table.T if models_in == "rows" else table         
    t = t.astype(float)
    if how == "target":
        score = (t - 0.80).abs()
        gap = score.sub(score.min(axis=1), axis=0)
        fmt = "{:.3f}"
    elif how == "max":
        best = t.max(axis=1)
        gap = t.rsub(best, axis=0).div(best, axis=0)
        fmt = "{:.1%}"
    else:
        best = t.min(axis=1)
        gap = t.sub(best, axis=0).div(best, axis=0)
        fmt = "{:.1%}"
    out = gap.mean().sort_values().rename("mean gap to best").to_frame()
    return out.style.highlight_min(props=BEST).format(fmt)

print("current PARAMS:", PARAMS)


current PARAMS: {'learning_rate': 0.05, 'num_leaves': 15, 'min_child_samples': 50, 'subsample': 0.8, 'subsample_freq': 1, 'colsample_bytree': 0.8, 'verbose': -1}


In [2]:
auctions_all = ridge.prepare_auctions(build_auctions(product="mFRR"), ("daily", "weekly"))
feats_ref = add_other_direction_features(
    add_lag_features(add_model_features(build_features(auctions_all))), auctions_all
)
FEATURE_COLS_REF = ridge.model_feature_columns(feats_ref) + LAG_FEATURES + OTHER_DIRECTION_FEATURES

VAL_START, VAL_LAST_DAY = "2022-01-01", "2024-12-31"     # same as lgbm.ipynb
VAL_END = (pd.Timestamp(VAL_LAST_DAY) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
in_val = (auctions_all["gc"] >= pd.Timestamp(VAL_START, tz="UTC")) & (auctions_all["gc"] < pd.Timestamp(VAL_END, tz="UTC"))
evaluator = Evaluator(actuals=auctions_all, start_test=VAL_START, end_test=VAL_END)

print("inputs:", FEATURE_COLS_REF)
print(VAL_START, "to", VAL_LAST_DAY, ":", in_val.sum(), "auctions")

inputs: ['lag1', 'lag2', 'lag3', 'block_00_04', 'block_04_08', 'block_08_12', 'block_12_16', 'block_16_20', 'block_20_24', 'daytype_saturday', 'daytype_sunday_holiday', 'daytype_workday', 'mom1', 'mom2', 'lag_mean', 'lag_std', 'other_lag1', 'other_gap']
2022-01-01 to 2024-12-31 : 13064 auctions


## 1. Changing one parameter at a time
Current parameters:

| parameter | now | controls |
|---|---|---|
| `num_leaves` | 15 | complexity of each tree |
| `min_child_samples` | 50 | minimum auctions per leaf (higher = smoother, less confident quantiles) |
| `colsample_bytree` | 0.8 | share of the features each tree may use |
| `subsample` | 0.8 | share of the auctions each tree is built on |
| `reg_lambda` | 0 | penalty on large leaf values |

`learning_rate` stays at 0.05 (the number of trees is chosen on the grid 100 / 300 / 600 every month).

In [3]:
pd.DataFrame({"overrides": {name: str(o) for name, o in CONFIGS.items()}})

,overrides
default,{}
leaves_7,{'num_leaves': 7}
leaves_31,{'num_leaves': 31}
child_20,{'min_child_samples': 20}
child_100,{'min_child_samples': 100}
child_200,{'min_child_samples': 200}
colsample_05,{'colsample_bytree': 0.5}
colsample_10,{'colsample_bytree': 1.0}
subsample_06,{'subsample': 0.6}
lambda_1,{'reg_lambda': 1.0}


In [4]:
report, tuned = run_configs(CONFIGS, feats_ref, FEATURE_COLS_REF, evaluator, VAL_START, VAL_LAST_DAY)

default: running (current PARAMS)
leaves_7: running {'num_leaves': 7}
leaves_31: running {'num_leaves': 31}
child_20: running {'min_child_samples': 20}
child_100: running {'min_child_samples': 100}
child_200: running {'min_child_samples': 200}
colsample_05: running {'colsample_bytree': 0.5}
colsample_10: running {'colsample_bytree': 1.0}
subsample_06: running {'subsample': 0.6}
lambda_1: running {'reg_lambda': 1.0}
lambda_5: running {'reg_lambda': 5.0}


## 2. Compare

`mean`: mean pinball over the four groups.

`groups_better`: in how many groups the
configuration beats `default`.

In [5]:
summary = summarise(report)
groups = [c for c in summary.columns if c not in ("mean", "groups_better")]
display(summary.style.highlight_min(axis=0, subset=groups + ["mean"], props=BEST).format(
    {**{c: "{:.4f}" for c in groups}, "mean": "{:.4f}", "groups_better": "{:.0f}"}))
display(mean_gap_to_best(summary[groups], models_in="rows", how="min"))

,daily downwards,daily upwards,weekly downwards,weekly upwards,mean,groups_better
config,,,,,,
default,0.0670,0.0757,0.0353,0.0136,0.0479,0
lambda_1,0.0671,0.0757,0.0353,0.0137,0.0480,0
colsample_05,0.0673,0.0758,0.0354,0.0136,0.0480,1
lambda_5,0.0673,0.0759,0.0353,0.0137,0.0480,0
subsample_06,0.0675,0.0757,0.0355,0.0135,0.0481,1
colsample_10,0.0673,0.0759,0.0353,0.0137,0.0481,0
leaves_31,0.0675,0.0758,0.0353,0.0136,0.0481,0
leaves_7,0.0675,0.0764,0.0353,0.0136,0.0482,0
child_100,0.0675,0.0759,0.0364,0.0139,0.0484,0


,mean gap to best
config,
default,0.2%
colsample_05,0.4%
subsample_06,0.4%
lambda_1,0.4%
lambda_5,0.4%
leaves_31,0.4%
colsample_10,0.5%
leaves_7,0.6%
child_100,1.6%


In [ ]:
coverage = report.pivot_table(index="config", columns=["freq", "direction"], values="coverage_80")

def closest_to_80(col):
    d = (col - 0.80).abs()
    return [BEST if v == d.min() else "" for v in d]

display(coverage.loc[summary.index].style.apply(closest_to_80, axis=0).format("{:.3f}"))
display(mean_gap_to_best(coverage, models_in="rows", how="target"))

,mean gap to best
config,
child_200,0.017
lambda_5,0.025
leaves_7,0.025
lambda_1,0.025
default,0.029
colsample_05,0.029
subsample_06,0.029
colsample_10,0.032
child_100,0.033


In [ ]:
mae = report.pivot_table(index="config", columns=["freq", "direction"], values="mae")
display(mae.loc[summary.index].style.highlight_min(axis=0, props=BEST).format("{:.3f}"))
display(mean_gap_to_best(mae, models_in="rows", how="min"))

,mean gap to best
config,
subsample_06,0.4%
child_100,0.9%
default,0.9%
colsample_05,1.0%
lambda_1,1.1%
lambda_5,1.1%
leaves_31,1.1%
colsample_10,1.2%
leaves_7,1.3%


## 3. Bidding revenue

The 3 best configurations, based on the mean pinball loss, the default and the baseline. Mean revenue per year in CHF per MW
offered in every auction.

In [8]:
TOP = [c for c in summary.index if c != "default"][:3]
contracts = {"baseline": PreviousKnownAuctionBaseline().fit(auctions_all).predict(auctions_all[in_val])}
contracts.update({name: tuned[name] for name in ["default"] + TOP})

bids = bidding_summary(contracts, auctions_all)
revenue = bids.pivot_table(index=["freq", "direction"], columns=["model", "strategy"], values="revenue")
print("Mean revenue per year (CHF per MW offered)")
display(revenue.style.highlight_max(axis=1, props=BEST).format("{:,.0f}"))
print("Mean gap to the best revenue (best strategy of each model)")
display(mean_gap_to_best(revenue.T.groupby(level="model").max().T, models_in="columns", how="max"))

share = bids.pivot_table(index=["freq", "direction"], columns=["model", "strategy"], values="share")
print("Share of the perfect revenue")
display(share.style.highlight_max(axis=1, props=BEST).format("{:.1%}"))

Mean revenue per year (CHF per MW offered)


Mean gap to the best revenue (best strategy of each model)


,mean gap to best
model,
default,0.1%
lambda_1,1.0%
lambda_5,1.1%
colsample_05,1.7%
baseline,20.6%


Share of the perfect revenue


## 4. Recent-sample weighting

Recent auctions weigh more in training: the weight halves every `half_life_months` before the last training
auction. Compared with the default in which all auctions weigh the same.

In [9]:
# only the weighted variants are run here (3 walk-forwards)
w_report, w_contracts = run_variants({k: v for k, v in WEIGHTING.items() if k != "default"},
                                     feats_ref, FEATURE_COLS_REF, evaluator, VAL_START, VAL_LAST_DAY)
w_report = pd.concat([report[report["config"] == "default"], w_report], ignore_index=True)
w_contracts["default"] = tuned["default"]

summary_w = summarise(w_report)
groups = [c for c in summary_w.columns if c not in ("mean", "groups_better")]
display(summary_w.style.highlight_min(axis=0, subset=groups + ["mean"], props=BEST).format(
    {**{c: "{:.4f}" for c in groups}, "mean": "{:.4f}", "groups_better": "{:.0f}"}))
display(mean_gap_to_best(summary_w[groups], models_in="rows", how="min"))

half_life_6m: running {'half_life_months': 6}
half_life_12m: running {'half_life_months': 12}
half_life_24m: running {'half_life_months': 24}


,daily downwards,daily upwards,weekly downwards,weekly upwards,mean,groups_better
config,,,,,,
half_life_24m,0.0670,0.0752,0.0360,0.0133,0.0479,3
default,0.0670,0.0757,0.0353,0.0136,0.0479,0
half_life_12m,0.0672,0.0749,0.0365,0.0136,0.0481,2
half_life_6m,0.0678,0.0747,0.0377,0.0143,0.0486,1


,mean gap to best
config,
half_life_24m,0.7%
default,1.0%
half_life_12m,1.7%
half_life_6m,4.0%


In [10]:
coverage_w = w_report.pivot_table(index="config", columns=["freq", "direction"], values="coverage_80")
display(coverage_w.loc[summary_w.index].style.apply(closest_to_80, axis=0).format("{:.3f}"))
display(mean_gap_to_best(coverage_w, models_in="rows", how="target"))

bids_w = bidding_summary({"baseline": contracts["baseline"], **w_contracts}, auctions_all)
revenue_w = bids_w.pivot_table(index=["freq", "direction"], columns=["model", "strategy"], values="revenue")
display(revenue_w.style.highlight_max(axis=1, props=BEST).format("{:,.0f}"))
print("Mean gap to the best revenue (best strategy of each model)")
display(mean_gap_to_best(revenue_w.T.groupby(level="model").max().T, models_in="columns", how="max"))

,mean gap to best
config,
default,0.009
half_life_12m,0.011
half_life_24m,0.017
half_life_6m,0.033


Mean gap to the best revenue (best strategy of each model)


,mean gap to best
model,
default,0.0%
half_life_12m,1.5%
half_life_24m,1.7%
half_life_6m,3.2%
baseline,20.4%


## 5. Sensitivity to market changes

Effect of changing the training window to only the last 2 or 3 years instead of the whole history and year by year performance of the models.


In [11]:
win_report, win_contracts = run_variants({k: v for k, v in WINDOWS.items() if k != "default"},
                                         feats_ref, FEATURE_COLS_REF, evaluator, VAL_START, VAL_LAST_DAY)
win_report = pd.concat([report[report["config"] == "default"], win_report], ignore_index=True)
win_contracts["default"] = tuned["default"]

summary_win = summarise(win_report)
groups = [c for c in summary_win.columns if c not in ("mean", "groups_better")]
display(summary_win.style.highlight_min(axis=0, subset=groups + ["mean"], props=BEST).format(
    {**{c: "{:.4f}" for c in groups}, "mean": "{:.4f}", "groups_better": "{:.0f}"}))
display(mean_gap_to_best(summary_win[groups], models_in="rows", how="min"))

last_2_years: running {'train_years': 2}
last_3_years: running {'train_years': 3}


,daily downwards,daily upwards,weekly downwards,weekly upwards,mean,groups_better
config,,,,,,
default,0.0670,0.0757,0.0353,0.0136,0.0479,0
last_3_years,0.0669,0.0756,0.0356,0.0144,0.0482,2
last_2_years,0.0675,0.0755,0.0371,0.0143,0.0486,1


,mean gap to best
config,
default,0.1%
last_3_years,1.7%
last_2_years,2.8%


In [12]:
# MAE year by year: baseline, default, and the training variants
models = {"baseline": contracts["baseline"], **w_contracts, **win_contracts}
years = per_year_report(models, auctions_all, pd.Timestamp(VAL_START).year, pd.Timestamp(VAL_LAST_DAY).year)
mae_years = years.pivot_table(index=["freq", "direction", "year"], columns="model", values="mae")
display(mae_years.style.highlight_min(axis=1, props=BEST).format("{:.3f}"))
display(mean_gap_to_best(mae_years, models_in="columns", how="min"))

,mean gap to best
model,
last_3_years,1.8%
half_life_24m,1.9%
default,2.2%
half_life_12m,3.2%
last_2_years,4.0%
half_life_6m,5.9%
baseline,18.3%


In [14]:
bids_win = bidding_summary({"baseline": contracts["baseline"], **win_contracts}, auctions_all)
revenue_win = bids_win.pivot_table(index=["freq", "direction"], columns=["model", "strategy"], values="revenue")
display(revenue_win.style.highlight_max(axis=1, props=BEST).format("{:,.0f}"))
print("Mean gap to the best revenue (best strategy of each model)")
display(mean_gap_to_best(revenue_win.T.groupby(level="model").max().T, models_in="columns", how="max"))

Mean gap to the best revenue (best strategy of each model)


,mean gap to best
model,
default,0.5%
last_2_years,1.1%
last_3_years,1.3%
baseline,20.9%


## Conclusion

All the comparisons are on the validation period 2022–2024, with the rules fixed in advance: a change is
kept only if it lowers the mean pinball loss (q10, q50, q90) and is better in at least 3 of the 4 groups,
without lowering the bidding revenue.

- **Hyper-parameters: no change.** No configuration beats the default (mean pinball 0.0479), and none
  is better in more than 1 of the 4 groups. All configurations are within 2% of the default, so
   the model is not sensitive to these parameters. 
- **Recent-sample weighting: not kept.** `half_life_24m` ties with the default on the mean pinball and
  is slightly better in 3 groups (daily upwards −0.7%, weekly upwards −2%), but worse in weekly
  downwards (+2%). Shorter half-lives help daily upwards (the group that changed most) and hurt the weekly
  groups: down-weighting old auctions amounts to training on fewer data. Coverage gets further from 0.80
  with every half-life, and the default has the highest revenue in all 4 groups.
- **Training window: not kept.** Training on the last 2 or 3 years only helps the weekly groups a little
  but hurts the daily ones, and no variant is the best in every year of any group. On the MAE year by year, `last_3_years` is slightly
  closer to the best than the default, but on the revenue the default is the
  closest to the best and the best in both daily groups.
- **Sensitivity to market changes.** The LightGBM beats the baseline in every year (2022, 2023, 2024)
  and every group without any special treatment of recent data.
- **Kept: the default** current `PARAMS`, whole history, all auctions with the same weight, one
  configuration for all groups. The gains of the project come from the features and from the bidding strategy, not from the
  hyper-parameters.